# Animating Bee Visitation Rate in a minute at different time intervals

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from datetime import datetime, timedelta
from IPython.display import HTML

In [ ]:
%matplotlib notebook
def load_dataset(filepath):
    with open(filepath, "rb") as f:
        results = pickle.load(f)
    return pd.DataFrame(results)

def extract_visit_data(df, start_time_str):
    sim_start_time = datetime.strptime("09:00:00", "%H:%M:%S")
    t_start = int((datetime.strptime(start_time_str, "%H:%M:%S") - sim_start_time).total_seconds())
    t_end = t_start + 60
    window_minutes = 1.0

    plant_df = df[
        (df["Agent Type"] == "plant") & (df["Step"] >= t_start) & (df["Step"] < t_end)
    ].copy()

    if isinstance(plant_df["Bee Positions"].iloc[0], str):
        plant_df["Bee Positions"] = plant_df["Bee Positions"].apply(eval)
    plant_df["x"] = plant_df["Bee Positions"].apply(lambda pos: pos[0])
    plant_df["Plant visit count"] = pd.to_numeric(plant_df["Plant visit count"], errors="coerce")
    plant_df["plant_id"] = plant_df["Bee Positions"].apply(lambda pos: tuple(pos))

    visit_stats = (
        plant_df.groupby(["RunId", "x", "plant_id"])["Plant visit count"]
        .agg(["max", "min"])
        .reset_index()
    )

    visit_stats["new_visits"] = visit_stats["max"] - visit_stats["min"]

    visits_per_x_run = (
        visit_stats.groupby(["RunId", "x"])["new_visits"]
        .sum()
        .reset_index()
    )

    visits_per_x_run["visits_per_minute"] = visits_per_x_run["new_visits"] / window_minutes
    return visits_per_x_run

def precompute_visit_counts(interval_data_dict, df_sample, interval=600):
    precomputed_visit_counts = {}
    all_visits = []

    start_time_seconds = 0
    end_time_seconds = df_sample["Step"].max() + 1

    for label, df in interval_data_dict.items():
        precomputed_visit_counts[label] = {}
        for current_time_seconds in range(start_time_seconds, end_time_seconds, interval):
            current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
            current_time_str = current_time.strftime("%H:%M:%S")
            visit_counts = extract_visit_data(df, current_time_str)
            precomputed_visit_counts[label][current_time_seconds] = visit_counts
            all_visits.extend(visit_counts["visits_per_minute"].values)

    global_ymin = 0
    global_ymax = max(all_visits) * 1.1 if all_visits else 1

    return precomputed_visit_counts, start_time_seconds, end_time_seconds, global_ymin, global_ymax

def animate_visits_precomputed(precomputed_visit_counts, start_time_seconds, end_time_seconds, global_ymin, global_ymax, interval=600):
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.set_theme(style="whitegrid")
    keys_list = list(precomputed_visit_counts.keys())
    num_simulations = len(next(iter(precomputed_visit_counts[keys_list[0]].values()))["RunId"].unique())

    def update(frame_idx):
        ax.clear()
        current_time_seconds = start_time_seconds + frame_idx * interval
        current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
        time_str_for_title = current_time.strftime("%H:%M")

        for label in precomputed_visit_counts.keys():
            visit_counts = precomputed_visit_counts[label][current_time_seconds]
            sns.lineplot(data=visit_counts, x="x", y="visits_per_minute", errorbar=("ci", 95), label=label, ax=ax)

        ax.set_ylim(global_ymin, global_ymax)
        ax.set_title(
            f"Mean number of bee visits in one-minute window starting at {time_str_for_title} across {num_simulations} simulations (95% CI)"
        )
        ax.set_xlabel("Polytunnel length (m)")
        ax.set_ylabel("Number of Bee Visits in One Minute Window")
        ax.grid(True)
        ax.legend()

    total_frames = (end_time_seconds - start_time_seconds) // interval
    anim = animation.FuncAnimation(fig, update, frames=total_frames, interval=500, repeat=False)
    return anim, HTML(anim.to_jshtml())


In [ ]:
# Load datasets
# df_pure = load_dataset("batch_results_pure_levy.pkl")
# df_truncated = load_dataset("batch_results_truncated_levy.pkl")
# df_composite = load_dataset("batch_results_composite_levy.pkl")
df_follow = load_dataset("batch_results_biased_composite.pkl")
df_mybehavior = load_dataset("batch_results_my_behavior.pkl")
interval_data_dict = {
    # "pure_levy": df_pure,
    # "truncated_levy": df_truncated,
    # "composite_levy": df_composite,
    "biased_composite_levy": df_follow,
    "my_bee_behavior": df_mybehavior 
}

# Precompute 10-minute averages
precomputed_visit_counts, start_sec, end_sec, global_ymin, global_ymax = precompute_visit_counts(interval_data_dict, df_mybehavior, interval=600)

# del df_pure, df_truncated, df_composite  # Free memory
del df_follow, df_mybehavior  # Free memory


# produce the animation
anim, html_obj = animate_visits_precomputed(
    precomputed_visit_counts, start_sec, end_sec, global_ymin, global_ymax, interval=600
)
html_obj

In [ ]:
# Saving the animation to HTML file
with open("bee_visitation_animation.html", "w") as f:
    f.write(anim.to_jshtml())

In [ ]:
# For visualization in a web browser
import webbrowser
webbrowser.open("bee_visitation_animation.html")

# Animate Nectar amount and bee visitation rate at different intervals of the simulation

In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns
from datetime import datetime, timedelta
from IPython.display import HTML

In [ ]:
def load_dataset(filepath):
    with open(filepath, "rb") as f:
        results = pickle.load(f)
    return pd.DataFrame(results)

def extract_visit_data(df, start_time_str):
    sim_start_time = datetime.strptime("09:00:00", "%H:%M:%S")
    t_start = int((datetime.strptime(start_time_str, "%H:%M:%S") - sim_start_time).total_seconds())
    t_end = t_start + 60  # 60-second window
    window_minutes = 1.0

    plant_df = df[
        (df["Agent Type"] == "plant") & (df["Step"] >= t_start) & (df["Step"] < t_end)
    ].copy()

    if isinstance(plant_df["Bee Positions"].iloc[0], str):
        import ast
        plant_df["Bee Positions"] = plant_df["Bee Positions"].apply(ast.literal_eval)

    plant_df["x"] = plant_df["Bee Positions"].apply(lambda pos: pos[0])
    plant_df["Plant visit count"] = pd.to_numeric(plant_df["Plant visit count"], errors="coerce")
    plant_df["Plant nectar amount"] = pd.to_numeric(plant_df["Plant nectar amount"], errors="coerce")
    plant_df["plant_id"] = plant_df["Bee Positions"].apply(lambda pos: tuple(pos))

    # Compute bee visit counts
    visit_stats = (
        plant_df.groupby(["RunId", "x", "plant_id"])["Plant visit count"]
        .agg(["max", "min"])
        .reset_index()
    )
    visit_stats["new_visits"] = visit_stats["max"] - visit_stats["min"]

    visits_per_x_run = (
        visit_stats.groupby(["RunId", "x"])["new_visits"]
        .sum()
        .reset_index()
    )
    visits_per_x_run["visits_per_minute"] = visits_per_x_run["new_visits"] / window_minutes

    # Total nectar per column (not per plant), per simulation
    nectar_per_column = (
        plant_df.groupby(["RunId", "x"])["Plant nectar amount"]
        .sum()
        .reset_index()
        .rename(columns={"Plant nectar amount": "total_nectar_per_column"})
    )

    return visits_per_x_run, nectar_per_column

def precompute_visit_counts(interval_data_dict, df_sample, interval=600):
    precomputed_data = {}
    all_visits = []

    start_time_seconds = 0
    end_time_seconds = df_sample["Step"].max() + 1

    for label, df in interval_data_dict.items():
        precomputed_data[label] = {}
        for current_time_seconds in range(start_time_seconds, end_time_seconds, interval):
            current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
            current_time_str = current_time.strftime("%H:%M:%S")
            visit_df, nectar_df = extract_visit_data(df, current_time_str)
            # Normalize nectar by number of 10s intervals in 10 minutes (i.e., 600s / 10s = 60)
            nectar_df["total_nectar_per_column"] /= 6
            precomputed_data[label][current_time_seconds] = (visit_df, nectar_df)
            all_visits.extend(visit_df["visits_per_minute"].values)

    global_ymin = 0
    global_ymax = max(all_visits) * 1.1 if all_visits else 1

    return precomputed_data, start_time_seconds, end_time_seconds, global_ymin, global_ymax

def animate_visits_with_nectar(precomputed_data, start_time_seconds, end_time_seconds, global_ymin, global_ymax, interval=600):
    fig, ax1 = plt.subplots(figsize=(12, 6))
    sns.set_theme(style="whitegrid")

    ax2 = ax1.twinx()
    keys_list = list(precomputed_data.keys())
    num_simulations = len(next(iter(precomputed_data[keys_list[0]].values()))[0]["RunId"].unique())

    def update(frame_idx):
        ax1.clear()
        ax2.clear()

        current_time_seconds = start_time_seconds + frame_idx * interval
        current_time = datetime.strptime("09:00:00", "%H:%M:%S") + timedelta(seconds=current_time_seconds)
        time_str_for_title = current_time.strftime("%H:%M")

        for i, label in enumerate(precomputed_data.keys()):
            visit_df, nectar_df = precomputed_data[label][current_time_seconds]

            # Plot visitation rate (left axis)
            sns.lineplot(data=visit_df, x="x", y="visits_per_minute", ax=ax1, label=f"{label} visits")

            # Plot average nectar (right axis, normalized by 6)
            nectar_avg = (
                nectar_df.groupby("x")["total_nectar_per_column"]
                .mean()
                .reset_index()
            )

            offset = (i - 0.5) * 0.4
            ax2.bar(nectar_avg["x"] + offset,
                    nectar_avg["total_nectar_per_column"],
                    width=0.6,
                    alpha=0.3,
                    label=f"{label} nectar")

        ax1.set_ylim(global_ymin, global_ymax)
        ax1.set_title(f"Bee Visits and Nectar Levels at {time_str_for_title} across {num_simulations} simulations")
        ax1.set_xlabel("Polytunnel length (m)")
        ax1.set_ylabel("Bee Visits / min")
        ax2.set_ylabel("Avg Nectar per Column (normalized)")

        ax2.yaxis.set_label_position("right")
        ax2.yaxis.tick_right()
        ax1.tick_params(axis='y', labelcolor='black')
        ax2.tick_params(axis='y', labelcolor='black')
        ax1.grid(True)
        ax1.legend(loc="upper left")
        ax2.legend(loc="upper right")

    total_frames = (end_time_seconds - start_time_seconds) // interval
    anim = animation.FuncAnimation(fig, update, frames=total_frames, interval=500, repeat=False)
    return anim, HTML(anim.to_jshtml())


In [ ]:
# Load datasets
df_follow = load_dataset("batch_results_biased_composite.pkl")
df_mybehavior = load_dataset("batch_results_my_behavior.pkl")

interval_data_dict = {
    "biased_composite_levy": df_follow,
    "my_bee_behavior": df_mybehavior
}

# Precompute at 10-minute intervals (600 seconds)
precomputed_data, start_sec, end_sec, global_ymin, global_ymax = precompute_visit_counts(
    interval_data_dict, df_mybehavior, interval=600
)

# Free memory
del df_follow, df_mybehavior

# Display animation
anim, html_obj = animate_visits_with_nectar(
    precomputed_data, start_sec, end_sec, global_ymin, global_ymax, interval=600
)
html_obj


In [ ]:
# Saving the animation to HTML file
with open("bee_visitation_animation.html", "w") as f:
    f.write(anim.to_jshtml())

In [ ]:
# For visualization in a web browser
import webbrowser
webbrowser.open("bee_visitation_animation.html")